<a href="https://colab.research.google.com/github/DHRUVCHARNE/AI-Learn-Notebooks/blob/main/gpt_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-06-14 13:14:58--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-06-14 13:14:59 (22.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [ ]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [ ]:
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [ ]:
block_size=24
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1])

In [ ]:
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
  context=x[:t+1]
  target=y[t]
  print(f'when input is {context}  the target: {target}\n')

when input is tensor([18])  the target: 47

when input is tensor([18, 47])  the target: 56

when input is tensor([18, 47, 56])  the target: 57

when input is tensor([18, 47, 56, 57])  the target: 58

when input is tensor([18, 47, 56, 57, 58])  the target: 1

when input is tensor([18, 47, 56, 57, 58,  1])  the target: 15

when input is tensor([18, 47, 56, 57, 58,  1, 15])  the target: 47

when input is tensor([18, 47, 56, 57, 58,  1, 15, 47])  the target: 58

when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])  the target: 47

when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])  the target: 64

when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64])  the target: 43

when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43])  the target: 52

when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52])  the target: 10

when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10])  the target: 0

when input is tensor([

In [ ]:
torch.manual_seed(1337)
batch_size=6
block_size=12
def get_batch(split):
  data = train_data if split =='train' else val_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y
xb,yb=get_batch('train')
print('inputs')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('--------------------')
for b in range(batch_size): # batch dimension
  for t in range(block_size): # time dimension
       context=xb[b,:t+1]
       target=yb[b,t]
       print(f"when input is {context.tolist()} the target: {target}")

inputs
torch.Size([6, 12])
tensor([[54, 43, 56, 53, 59, 57,  1, 58, 56, 39, 47, 58],
        [17, 31, 32, 17, 30, 10,  0, 15, 53, 51, 43,  6],
        [33, 23, 17,  1, 34, 21, 26, 15, 17, 26, 32, 21],
        [30, 27, 25, 17, 27, 10,  0, 32, 46, 53, 59,  1],
        [ 1, 57, 58, 53, 52, 43,  1, 48, 59, 45, 57,  1],
        [45, 43, 58,  1, 58, 46, 43, 43,  1, 45, 53, 52]])
targets:
torch.Size([6, 12])
tensor([[43, 56, 53, 59, 57,  1, 58, 56, 39, 47, 58, 53],
        [31, 32, 17, 30, 10,  0, 15, 53, 51, 43,  6,  1],
        [23, 17,  1, 34, 21, 26, 15, 17, 26, 32, 21, 27],
        [27, 25, 17, 27, 10,  0, 32, 46, 53, 59,  1, 42],
        [57, 58, 53, 52, 43,  1, 48, 59, 45, 57,  1, 39],
        [43, 58,  1, 58, 46, 43, 43,  1, 45, 53, 52, 43]])
--------------------
when input is [54] the target: 43
when input is [54, 43] the target: 56
when input is [54, 43, 56] the target: 53
when input is [54, 43, 56, 53] the target: 59
when input is [54, 43, 56, 53, 59] the target: 57
when input is [

In [ ]:
print(xb)

tensor([[54, 43, 56, 53, 59, 57,  1, 58, 56, 39, 47, 58],
        [17, 31, 32, 17, 30, 10,  0, 15, 53, 51, 43,  6],
        [33, 23, 17,  1, 34, 21, 26, 15, 17, 26, 32, 21],
        [30, 27, 25, 17, 27, 10,  0, 32, 46, 53, 59,  1],
        [ 1, 57, 58, 53, 52, 43,  1, 48, 59, 45, 57,  1],
        [45, 43, 58,  1, 58, 46, 43, 43,  1, 45, 53, 52]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    # each token directly reads off the logits for the next token
    # from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
  def forward(self,idx,targets=None):
    # idx and targets are both (B,T) tensor of integers
    logits=self.token_embedding_table(idx) #(B,T,C)
    if targets is None:
        loss = None
    else:
        B,T,C = logits.shape
        logits_flat = logits.view(B*T, C)
        targets_flat = targets.view(B*T)
        loss = F.cross_entropy(logits_flat, targets_flat)
    return logits,loss
  def generate(self,idx,max_new_tokens):
    # idx is (B,T) array of indices of the current context
    for _ in range(max_new_tokens):
      # get predictions
      logits,loss = self(idx)
      # focus only on the last time step
      logits = logits[:,-1,:] # becomes (B,C)
      # apply softmax to the probabilities
      probs = F.softmax(logits,dim=-1) # (B,C)
      # sample from the distribution
      idx_next = torch.multinomial(probs,num_samples=1) # (B,1)
      # append the sampled index to the running sequence
      idx = torch.cat((idx,idx_next),dim=1) # (B,T+1)
    return idx
m=BigramLanguageModel(vocab_size)
logits,loss=m(xb,yb)
print(logits.shape)
print(loss)
idx= torch.zeros((1,1),dtype = torch.long)
print(decode(m.generate(idx,max_new_tokens=100)[0].tolist()))

torch.Size([6, 12, 65])
tensor(4.6786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
# create a pytorch optimizer
optimizer= torch.optim.AdamW(m.parameters(),lr=1e-4)


In [ ]:
batch_size = 32
for steps in range(2000):
  # sample a batch of data
  xb,yb = get_batch('train')
  # evaluate the loss
  logits,loss = m(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
  if steps%1000==0:
    print(loss.item())

4.790748119354248
4.610840320587158


In [ ]:
idx= torch.zeros((1,1),dtype = torch.long)
print(decode(m.generate(idx,max_new_tokens=100)[0].tolist()))


gb
LsuF&ubfNoxqftcHMlaG&natqKXKIca!a?qAU,iwfta-KVOl3fNrYCJAUAGxQd&3MgqUO C qH-o?qf szhsk-IuxEQ;.SqmJ


**The Mathematical Trick of Self Attention**


In [ ]:
# Consider the following toy example
torch.manual_seed(1337)
B,T,C=4,8,32 # Batch, Time, Channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 32])

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] # (t,C)
    xbow[b,t]=torch.mean(xprev,0)

In [ ]:
xbow

tensor([[[ 1.8077e-01, -6.9988e-02, -3.5962e-01,  ..., -8.0164e-01,
           1.5236e+00,  2.5086e+00],
         [-2.4116e-01, -1.6063e-01,  3.2526e-01,  ...,  3.6581e-01,
           1.5667e+00,  1.0527e+00],
         [-4.3893e-01,  9.2179e-02,  1.9971e-01,  ...,  9.8210e-02,
           7.1071e-01,  5.6531e-01],
         ...,
         [-9.8921e-01,  1.3417e-01,  2.8014e-01,  ...,  3.4951e-01,
           6.1414e-01,  1.2510e-01],
         [-8.2062e-01,  6.6077e-02,  4.9662e-01,  ...,  3.5242e-01,
           4.4703e-01,  5.0332e-02],
         [-7.9077e-01,  3.0213e-02,  4.3624e-01,  ...,  6.9874e-02,
           3.2521e-01,  1.7912e-01]],

        [[ 4.5618e-01, -1.0917e+00, -8.2073e-01,  ...,  5.1187e-02,
          -6.5764e-01, -2.5729e+00],
         [ 2.3859e-01, -4.2831e-02, -1.0349e+00,  ...,  4.1852e-01,
          -9.0388e-01, -6.2984e-01],
         [ 8.9262e-01, -1.0170e-01, -5.0905e-01,  ...,  6.4184e-02,
          -2.4146e-01, -6.8638e-01],
         ...,
         [ 5.6778e-01,  3

In [ ]:
wei = torch.tril(torch.ones(T,T))
wei = wei/wei.sum(1,keepdim=True)
xbow2 = wei @ x # (B,T,T) @ (B,T,C) ----> (B,T,C)
torch.allclose(xbow,xbow2,atol=1e-6)


True

In [ ]:
torch.manual_seed(42)
a=torch.tril(torch.ones(3,3))
a = a/torch.sum(a,1,keepdim=True)
b= torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# version 3 Use Softmax
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0,float('-inf'))
wei = F.softmax(wei,dim=-1)
xbow3 = wei @ x
torch.allclose(xbow,xbow3,atol=1e-6)

True

In [ ]:
# version 4 use self attention single head
torch.manual_seed(1357)
B,T,C=4,8,64 # Batch , Time , Channels
x = torch.randn(B,T,C)
# Let's see a single head perform self attention
head_size = 32
key = nn.Linear(C,head_size,bias=False)
query = nn.Linear(C,head_size,bias=False)
value = nn.Linear(C,head_size,bias=False)
k = key(x) # (B,T,head_size)
q = query(x) # (B,T,head_size)
wei = q @ k.transpose(-2,-1) # (B,T,head_size) @ (B,head_size,T) = (B,T,T)
tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril==0,float('-inf'))
wei = F.softmax(wei,dim=-1)
v = value(x)
out = wei @ v #(B,T,T) @ (B,T,C) = (B,T,C)


In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2,-1) * head_size**-0.5;

In [ ]:
k.var()

tensor(0.9957)

In [ ]:
wei.var()

tensor(1.0256)

In [ ]:
torch.softmax(torch.tensor([-0.1,0.2,0.3,0.4,0.2]),dim=-1)

tensor([0.1462, 0.1973, 0.2181, 0.2410, 0.1973])